# AI for Coding and Data Analysis: Starter notebook

Workshop, October 14, 2026 · Inna Kouper, Digital Scholarship and Data Services, Sheridan Libraries

**Before you start:** File → *Save a copy in Drive*, then add your name to the title. You'll work in your own copy.

**About the data:** a *synthetic* export from a fictional "Research Data Practices" survey, built for teaching. It has the problems real survey exports have. Please don't upload your own data during the workshop.

**How to use this notebook:** each section has a suggested prompt for Gemini (click the ✦ Gemini button or open the Gemini panel). Run cells with ▶ or Shift+Enter.

## 1. Get the data
Run the cell below. It downloads the survey exports and the codebook into this session's temporary storage (see the 📁 Files panel on the left).
These files disappear when the runtime resets, which is why downloading them is part of the notebook.

In [ ]:
# Download the survey exports and codebook into this runtime.
import urllib.request

BASE = "https://raw.githubusercontent.com/jhu-data-services/ai-coding-data-analysis/main/data/"
FILES = ["data_practices_survey_2026-09-30.xlsx",
         "data_practices_survey_2026-10-09.xlsx",
         "data_practices_survey_codebook.xlsx"]

for f in FILES:
    urllib.request.urlretrieve(BASE + f, f)
    print("downloaded", f)

## 2. Load and look
Look before you change anything.

> **Suggested prompt:** `data_practices_survey_2026-09-30.xlsx is a Qualtrics export. Row 1 has the column names; rows 2 and 3 hold question text and import IDs, so skip them. Load it with pandas into a DataFrame called raw. Show the number of rows, the column names, and the first 5 rows. Don't modify the file.`

> **Then try:** `For each column in raw, show its type and how many values are missing. List the unique values of Status and Finished.`

## 3. Clean, one step at a time
Ask for one step per cell, and ask Gemini to explain what it wrote. Steps we'll take in the demo:
1. Drop identifying and empty metadata columns (IP address, location, recipient fields).
2. Keep only real, finished responses (`Status` is "IP Address" and `Finished` is True).
3. Remove duplicate submissions (same IP address and same answers).
4. Turn the satisfaction labels in `Q7` into numbers 1-5 in a new column called `satisfaction`.
5. Turn the free-text hours in `Q4` into numbers in a new column called `hours`, using a rule you write down.

Call the cleaned DataFrame `df`.

### Catch-up cell
If your cleaning went sideways, run this cell. It makes the same decisions as the demo and gives you a clean DataFrame called `df`, so you can continue with the next sections.

In [ ]:
import re
import pandas as pd

SATISFACTION = {"Extremely dissatisfied": 1, "Somewhat dissatisfied": 2,
                "Neither satisfied nor dissatisfied": 3,
                "Somewhat satisfied": 4, "Extremely satisfied": 5}

def hours_to_number(text):
    """Documented rule: '5' -> 5; '4-6' -> 5 (midpoint); '10+' -> 10; 'less than 1' -> 0.5;
    '~5', 'about 5', '5ish', '5 hrs' -> 5; answers with no number ('depends') -> missing."""
    if pd.isna(text):
        return None
    t = str(text).strip().lower()
    if t in ("less than 1", "<1"):
        return 0.5
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", t)
    if m:
        return (int(m[1]) + int(m[2])) / 2
    m = re.search(r"\d+(\.\d+)?", t)
    return float(m[0]) if m else None

def clean_export(path):
    raw = pd.read_excel(path, skiprows=[1, 2])                  # skip question-text and import-ID rows
    df = raw[(raw["Status"] == "IP Address") & (raw["Finished"] == True)]   # drop previews, incomplete
    answers = [c for c in df.columns if c.startswith("Q")]
    df = df.drop_duplicates(subset=["IPAddress"] + answers)     # same person submitting twice
    df = df.drop(columns=["IPAddress", "LocationLatitude", "LocationLongitude",
                          "RecipientLastName", "RecipientFirstName", "RecipientEmail",
                          "ExternalReference"])
    df["hours"] = df["Q4"].map(hours_to_number)
    df["satisfaction"] = df["Q7"].map(SATISFACTION)
    return df.reset_index(drop=True)

df = clean_export("data_practices_survey_2026-09-30.xlsx")
print(len(df), "valid responses")

## 4. Your turn: how do people share data?
Question `Q5` allowed several answers, joined with semicolons (for example `Within my research team;By request`).

> **Suggested prompt:** `In df, the column Q5 holds multiple answers separated by semicolons. Split it and count how many respondents chose each option.`

**Done when:** you see five options with a count for each. Then change one word in your prompt to do the same for `Q6`.

## 5. Summarize and chart

> **Suggested prompts:** `Make a table of the number of respondents, mean hours, and mean satisfaction by discipline (Q1).` · `Make a bar chart of mean hours by primary tool (Q3).` · `Sort the bars, label the axes, and use colorblind-friendly colors.`

## 6. Themes in the comments
`Q8` holds optional free-text comments.

> **Suggested prompt:** `Read the non-empty comments in Q8 and group them into 6-8 themes. For each theme give a short name, a one-sentence description, and the number of comments. Show which comments you put in each theme.`

Then read some comments yourself. Do the themes hold up?

## 7. Check: restart and run all
**Runtime → Restart session and run all.** When it finishes, this cell should report the numbers below with no errors.

In [ ]:
# Known values for the 2026-09-30 export
print("Valid responses:", len(df))
if "hours" in df.columns:
    print("Mean hours per week cleaning data:", round(df["hours"].mean(), 1))
assert len(df) == 150, "Expected 150 valid responses: check the filtering and duplicate steps."
print("Checks passed.")

## 8. A new export arrives
The survey stayed open. `data_practices_survey_2026-10-09.xlsx` contains every earlier response plus the new ones.

> **Suggested prompt:** `Run the same cleaning on data_practices_survey_2026-10-09.xlsx and show how many valid responses there are.`

## 9. Document and share
> **Suggested prompt:** `Add a text cell above each code cell explaining in plain language what it does and why. At the top, add a header with the data source, date, tools used, AI assistance, and how to run this notebook.`

Read what it writes and correct anything that misstates a decision.